# Read file

In [8]:
import pandas as pd
import numpy as np

file_path = './dataset/observations_part_001.parquet'
df_parquet = pd.read_parquet(file_path)

print(f"Row Total: {df_parquet.shape[0]:,}")
print(f"Column Total: {df_parquet.shape[1]}")

Row Total: 480,000
Column Total: 59


# Data Info


In [9]:
df_parquet.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480000 entries, 0 to 479999
Data columns (total 59 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   activity_multiplier           480000 non-null  float32
 1   animal_residue_pct            480000 non-null  int32  
 2   battery_level_percent         477785 non-null  float32
 3   battery_maintenance_event     480000 non-null  int32  
 4   bin_index                     480000 non-null  int32  
 5   collection_delay_minutes      480000 non-null  float32
 6   collection_event              480000 non-null  int32  
 7   collection_threshold_percent  480000 non-null  float32
 8   college_calendar_closed       480000 non-null  int32  
 9   date_index                    480000 non-null  int32  
 10  day_of_week                   480000 non-null  int32  
 11  dominant_waste_code           480000 non-null  int32  
 12  event_code                    480000 non-nul

In [10]:
df_parquet.head()

,activity_multiplier,animal_residue_pct,battery_level_percent,battery_maintenance_event,bin_index,collection_delay_minutes,collection_event,collection_threshold_percent,college_calendar_closed,date_index,...,sensor_error_code,temperature_c,textile_pct,time_slot_code,timestamp_unix,true_fill_level_percent,waste_bulk_density_kg_m3,waste_entering_bin_kg,waste_volume_liter,weather_code
0,0.6984,4,96.482681,0,0,0.0,0,80.0,0,4,...,0,13.053044,6,0,1736013600,49.544697,226.006271,0.629307,2.784469,0
1,0.5626,0,92.744682,0,1,0.0,0,80.0,0,4,...,0,14.072392,1,0,1736013600,49.981346,251.556946,0.622490,2.474549,0
2,0.8924,1,96.756447,0,2,0.0,0,80.0,0,4,...,0,13.722422,4,0,1736013600,32.794777,232.091263,1.656849,7.138784,0
3,0.2134,3,95.200546,0,3,0.0,0,75.0,0,4,...,0,13.480151,4,0,1736013600,24.558718,199.116287,0.294287,1.477966,0
4,0.8924,0,92.272179,0,4,0.0,0,75.0,0,4,...,0,13.799287,3,0,1736013600,67.464378,222.365173,2.896791,13.027180,0


# Exploration One Bin

In [11]:
# Data Filtering for bin_index 0
df_par0 = df_parquet[df_parquet['bin_index'] == 0].copy()

# Urutkan berdasarkan waktu agar runut
df_par0 = df_par0.sort_values('timestamp_unix').reset_index(drop=True)

print(f"Total Record for Bin Index 0 in this file:  {df_par0.shape[0]} baris")

# See the min max for capacity
df_par0[['fill_level_percent', 'true_fill_level_percent']].describe()

Total Record for Bin Index 0 in this file:  96 baris


,fill_level_percent,true_fill_level_percent
count,96.000000,96.000000
mean,49.945301,49.883942
std,20.483332,20.424936
min,7.828559,7.332739
25%,31.914609,32.495759
50%,53.673677,53.687769
75%,65.756458,66.264305
max,81.204727,80.511337


## Ekstraksi Multi-Bin untuk Data Training (Mengambil 50 Tempat Sampah)

Karena 1 tempat sampah hanya berisi 96 baris (4 hari), kita mengambil 50 tempat sampah sekaligus agar model AI memiliki cukup variasi pola untuk dipelajari.

In [12]:
# Ambil data untuk 50 tempat sampah pertama (Bin 0 sampai 49)
df_multi_bin = df_parquet[df_parquet['bin_index'] < 50].copy()

# SANGAT PENTING: Urutkan berdasarkan ID tempat sampah dulu, baru berdasarkan waktu
df_multi_bin = df_multi_bin.sort_values(['bin_index', 'timestamp_unix']).reset_index(drop=True)

print(f"Total baris data gabungan 50 tempat sampah: {df_multi_bin.shape[0]:,} baris")

Total baris data gabungan 50 tempat sampah: 4,800 baris


# Feature Engineering (Membuat Lag Features & Target Prediksi)
Kita memilih kolom Waktu (`hour`, `day_of_week`) dan Kapasitas (`fill_level_percent`). 
Lalu, kita menyuruh pandas membuat kolom baru berisi riwayat kapasitas 1, 2, dan 3 jam ke belakang menggunakan metode `shift()`, yang dikelompokkan berdasarkan `bin_index` agar riwayat tempat sampah yang berbeda tidak saling tercampur.

In [13]:
# Pilih kolom esensial yang sesuai dengan data asli dari alat ESP32
df_clean = df_multi_bin[['bin_index', 'hour', 'day_of_week', 'fill_level_percent']].copy()

# Buat Fitur Input (Riwayat kapasitas sebelumnya)
# Groupby bin_index memastikan data bin 0 tidak tergeser masuk ke data bin 1
df_clean['kapasitas_1_jam_lalu'] = df_clean.groupby('bin_index')['fill_level_percent'].shift(1)
df_clean['kapasitas_2_jam_lalu'] = df_clean.groupby('bin_index')['fill_level_percent'].shift(2)
df_clean['kapasitas_3_jam_lalu'] = df_clean.groupby('bin_index')['fill_level_percent'].shift(3)

# Buat Target Output (Apa yang akan ditebak AI? -> Kapasitas 1 jam ke depan)
df_clean['target_kapasitas_jam_depan'] = df_clean.groupby('bin_index')['fill_level_percent'].shift(-1)

# Hapus baris yang berisi NaN (akibat proses pergeseran ke belakang/depan)
df_clean = df_clean.dropna().reset_index(drop=True)

# Hapus kolom bin_index karena AI tidak perlu tahu identitas tempat sampah saat dilatih
df_clean = df_clean.drop(columns=['bin_index'])

# Tampilkan hasil akhir tabel AI yang sudah sangat bersih
print(f"Total Baris Data Siap Training: {df_clean.shape[0]:,} baris")
df_clean.head(10)

Total Baris Data Siap Training: 4,488 baris


,hour,day_of_week,fill_level_percent,kapasitas_1_jam_lalu,kapasitas_2_jam_lalu,kapasitas_3_jam_lalu,target_kapasitas_jam_depan
0,3,6,50.850685,52.753166,50.935368,49.665939,53.736298
1,4,6,53.736298,50.850685,52.753166,50.935368,54.325520
2,5,6,54.325520,53.736298,50.850685,52.753166,53.611057
3,6,6,53.611057,54.325520,53.736298,50.850685,55.610485
4,7,6,55.610485,53.611057,54.325520,53.736298,56.233257
5,8,6,56.233257,55.610485,53.611057,54.325520,55.881794
6,9,6,55.881794,56.233257,55.610485,53.611057,58.494690
7,10,6,58.494690,55.881794,56.233257,55.610485,60.227432
8,11,6,60.227432,58.494690,55.881794,56.233257,59.262806
9,12,6,59.262806,60.227432,58.494690,55.881794,60.554939


# Ekspor ke CSV
Data yang sudah dinormalisasi dan diubah bentuknya menjadi *Time-to-Event* ini disimpan ke CSV. File inilah yang akan diunggah ke algoritma Machine Learning (Random Forest).

In [15]:
output_csv = 'data_training_siap_pakai.csv'
df_clean.to_csv(output_csv, index=False)
print(f"Data telah berhasil disimpan menjadi '{output_csv}'")

Data telah berhasil disimpan menjadi 'data_training_siap_pakai.csv'
